## Attention Visualization

In [22]:
import torch
import altair as alt
import pandas as pd
from config import get_config, get_weights_file_path
from train import get_dataset, get_model, greedy_decode
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


1. Load configuration and model

In [23]:
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataset(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

# Load the pretrained weights (automatically finds latest, or defaults to specific epoch)
if config['preload'] == 'latest':
   model_folder = Path(config['model_folder'])
   model_files = list(model_folder.glob(f"{config['model_basename']}*.pt"))
   if model_files:
      model_filename = str(sorted(model_files)[-1])
      print(f"Loading latest model: {model_filename}")
   else:
      print("No weights found. Make sure to train the model first.")
      model_filename = None
else:
   model_filename = get_weights_file_path(config, config['preload'] if config['preload'] else "09")

if model_filename and Path(model_filename).exists():
   state = torch.load(model_filename)
   model.load_state_dict(state['model_state_dict'])
   print("Weights loaded successfully!")


Loaded tokenizer for en from D:\Krishan\Python\GEN AI\Transformer\tokenizer_en.json
Loaded tokenizer for hi from D:\Krishan\Python\GEN AI\Transformer\tokenizer_hi.json
Weights loaded successfully!


2. Define helper functions

In [24]:
model.eval()

def load_next_batch():
   # Load a sample batch from the validation set
   batch = next(iter(val_dataloader))
   encoder_input = batch["encoder_input"].to(device)
   encoder_mask = batch["encoder_mask"].to(device)
   decoder_input = batch["decoder_input"].to(device)
   decoder_mask = batch["decoder_mask"].to(device)

   # Get tokens from IDs
   encoder_input_tokens = [tokenizer_src.id_to_token(idx.item()) for idx in encoder_input[0].cpu()]
   encoder_input_tokens = [t if t is not None else "[PAD]" for t in encoder_input_tokens]
    
   decoder_input_tokens = [tokenizer_tgt.id_to_token(idx.item()) for idx in decoder_input[0].cpu()]
   decoder_input_tokens = [t if t is not None else "[PAD]" for t in decoder_input_tokens]

   # check that the batch size is 1
   assert encoder_input.size(0) == 1, "Batch size must be 1 for validation"

   with torch.no_grad():
      model_out = greedy_decode(
         model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, config['seq_len'], device)
    
   return batch, encoder_input_tokens, decoder_input_tokens


In [25]:
def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
   return pd.DataFrame([
      (r, c, float(m[r, c]),
         "%.3d %s" % (r, row_tokens[r] if len(row_tokens) > r else "<blank>"),
         "%.3d %s" % (c, col_tokens[c] if len(col_tokens) > c else "<blank>"),
      )  for r in range(m.shape[0])
         for c in range(m.shape[1])
         if r < max_row and c < max_col
      ], columns=["row", "column", "value", "row_token", "col_token"],
   )

def get_attn_map(attn_type: str, layer: int, head: int):
   with torch.no_grad():
      if attn_type == "encoder":
         attn = model.encoder.layers[layer].attention_block.attention_scores
      elif attn_type == "decoder":
         attn = model.decoder.layers[layer].self_attention.attention_scores
      elif attn_type == "encoder-decoder":
         attn = model.decoder.layers[layer].cross_attention.attention_scores
      return attn[0, head].data.cpu()

In [26]:
def attn_map(attn_type, layer, head, row_tokens, col_tokens, max_sentence_len):
   df = mtx2df(
      get_attn_map(attn_type, layer, head),
      max_sentence_len,
      max_sentence_len,
      row_tokens,
      col_tokens)
   
   return (alt.Chart(data=df).mark_rect().encode(
         x=alt.X("col_token", axis=alt.Axis(title="")),
         y=alt.Y("row_token", axis=alt.Axis(title="")),
         color="value",
         tooltip=["row", "column", "value", "row_token", "col_token"],
      ).properties(height=400, width=400, title=f"Layer {layer} Head {head}")
      .interactive())

def get_all_attention_maps(attn_type: str, layers: list[int], heads: list[int], row_tokens: list, col_tokens, max_sentence_len: int):
   charts = []
   for layer in layers:
      rowCharts = []
      for head in heads:
         rowCharts.append(attn_map(attn_type, layer, head, row_tokens, col_tokens, max_sentence_len))
      charts.append(alt.hconcat(*rowCharts))
   return alt.vconcat(*charts)


3. Predict and Visualize

In [27]:
batch, encoder_input_tokens, decoder_input_tokens = load_next_batch()

# Display Source/Target cleanly
print(f'Source: {batch["source_text"][0]}')
print(f'Target: {batch["target_text"][0]}')

# Try finding length up to the first padding token, defaulting to sequence length if none found
try:
   sentence_len = encoder_input_tokens.index("[PAD]")
except ValueError:
   sentence_len = len(encoder_input_tokens)

Source: _ Game
Target: खेल (_ G) 


4. Test run

In [28]:
# Your config sets: "num_encoder_layers": 2, "num_decoder_layers": 3, "num_heads": 4
layers = [0, 1] # Valid layers based on your config.py
heads = [0, 1, 2, 3] # Valid heads based on your config.py

In [29]:
# 3a. Encoder Self-Attention Visualization
get_all_attention_maps("encoder", layers, heads, encoder_input_tokens, encoder_input_tokens, min(20, sentence_len))

alt.VConcatChart(...)

In [30]:
# 3b. Decoder Self-Attention Visualization
get_all_attention_maps("decoder", layers, heads, decoder_input_tokens, decoder_input_tokens, min(20, sentence_len))

alt.VConcatChart(...)

In [31]:
# 3c. Encoder-Decoder Cross-Attention Visualization
get_all_attention_maps("encoder-decoder", layers, heads, encoder_input_tokens, decoder_input_tokens, min(20, sentence_len))

alt.VConcatChart(...)